# SIREC — Fine-tuning de BETO (Fase D5) en Google Colab

Ajusta **BETO** (`dccuchile/bert-base-spanish-wwm-cased`) para las dos tareas del proyecto:
`categoria` (7 clases) y `urgencia` (3 niveles). Entrena **un modelo por tarea**.

**Protocolo (idéntico al baseline D4, para comparación directa):**
- **Test = 180 reportes gold** (`gold_kappa.csv`): doblemente etiquetados y adjudicados por consenso.
- **Train = 220 reales restantes + 900 sintéticos** (declarados; los sintéticos solo entrenan). Sin fuga.
- Pérdida ponderada por clase (class weights) por el desbalance real. Foco en **recall** de las clases
  críticas (`persona_en_riesgo`, `urgencia=alta`), según el criterio del evaluador.

**Antes de correr:** menú *Entorno de ejecución → Cambiar tipo de entorno → Acelerador por hardware = GPU (T4)*.

**Piso a superar (baseline D4, test gold):** categoría macro-F1 ≈ 0.65 (recall `persona_en_riesgo` 0.58);
urgencia macro-F1 ≈ 0.45 (recall `alta` 0.58).

## 1. Instalar dependencias

In [ ]:
!pip -q install "transformers>=4.40" "datasets>=2.19" "accelerate>=0.30" scikit-learn pandas

In [ ]:
import torch
print('GPU disponible:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('ADVERTENCIA: sin GPU. Activa GPU en Entorno de ejecución para que sea rápido.')

## 2. Subir los datos

Sube **tres archivos** desde tu carpeta `datos-modelo`:
`corpus_etiquetado.csv`, `corpus_sintetico.csv` y `gold_kappa.csv`.

In [ ]:
from google.colab import files
subidos = files.upload()  # selecciona los 3 CSV
import os
for f in ['corpus_etiquetado.csv', 'corpus_sintetico.csv', 'gold_kappa.csv']:
    print(('OK  ' if os.path.exists(f) else 'FALTA ') + f)

## 3. Código de entrenamiento (una función reutilizable)

In [ ]:
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import classification_report, f1_score, recall_score
from sklearn.utils.class_weight import compute_class_weight
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding)

MODELO_BASE = 'dccuchile/bert-base-spanish-wwm-cased'
CRITICO = {'categoria': 'persona_en_riesgo', 'urgencia': 'alta'}
SEED = 42


def norm(t):
    return ' '.join(str(t or '').lower().split())


def cargar(tarea, con_sintetico=True):
    """Test = 180 gold adjudicados; train = reales restantes (+ sintéticos). Sin fuga."""
    real = pd.read_csv('corpus_etiquetado.csv', encoding='utf-8-sig')
    real = real[real['origen'] == 'real'].copy()
    gold = pd.read_csv('gold_kappa.csv', encoding='utf-8-sig')
    textos_gold = {norm(t) for t in gold['texto']}
    tr = real[~real['texto'].map(norm).isin(textos_gold)].copy()
    te = gold.copy()
    print('Split gold: train=%d reales | test=%d gold' % (len(tr), len(te)))
    if con_sintetico:
        sint = pd.read_csv('corpus_sintetico.csv', encoding='utf-8-sig')
        tr = pd.concat([tr, sint], ignore_index=True)
        print('Entrenamiento aumentado con %d sintéticos (solo train).' % len(sint))
    return tr.reset_index(drop=True), te.reset_index(drop=True)


class TrainerPonderado(Trainer):
    """Trainer con pérdida ponderada por clase (desbalance del corpus real)."""
    def __init__(self, pesos, *a, **k):
        super().__init__(*a, **k)
        self.pesos = pesos

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')
        out = model(**inputs)
        loss = torch.nn.functional.cross_entropy(
            out.logits, labels, weight=self.pesos.to(out.logits.device))
        return (loss, out) if return_outputs else loss


def entrenar_tarea(tarea, con_sintetico=True, epocas=4, batch=16, salida=None):
    salida = salida or ('./beto_' + tarea)
    tr, te = cargar(tarea, con_sintetico)
    clases = sorted(pd.unique(pd.read_csv('corpus_etiquetado.csv', encoding='utf-8-sig')[tarea]))
    id2lab = {i: c for i, c in enumerate(clases)}
    lab2id = {c: i for i, c in id2lab.items()}
    tok = AutoTokenizer.from_pretrained(MODELO_BASE)

    def preparar(df):
        d = Dataset.from_pandas(pd.DataFrame({
            'text': df['texto'].astype(str).values,
            'labels': [lab2id[c] for c in df[tarea].astype(str).values]}))
        return d.map(lambda b: tok(b['text'], truncation=True, max_length=128), batched=True)

    ds_tr, ds_te = preparar(tr), preparar(te)
    y_tr = np.array([lab2id[c] for c in tr[tarea].astype(str).values])
    pesos = compute_class_weight('balanced', classes=np.arange(len(clases)), y=y_tr)
    pesos_t = torch.tensor(pesos, dtype=torch.float)

    modelo = AutoModelForSequenceClassification.from_pretrained(
        MODELO_BASE, num_labels=len(clases), id2label=id2lab, label2id=lab2id)

    def metricas(pred):
        y = pred.label_ids
        yhat = np.argmax(pred.predictions, axis=1)
        crit_id = lab2id.get(CRITICO[tarea])
        rec_crit = recall_score(y, yhat, labels=[crit_id], average='macro', zero_division=0) \
            if crit_id is not None else 0.0
        return {'macro_f1': f1_score(y, yhat, average='macro', zero_division=0),
                'recall_critico': rec_crit}

    ta = TrainingArguments(
        output_dir=salida, num_train_epochs=epocas,
        per_device_train_batch_size=batch, per_device_eval_batch_size=batch,
        learning_rate=2e-5, eval_strategy='epoch', save_strategy='epoch',
        load_best_model_at_end=True, metric_for_best_model='recall_critico',
        fp16=torch.cuda.is_available(), seed=SEED, logging_steps=20, report_to='none')

    trainer = TrainerPonderado(
        pesos_t, model=modelo, args=ta, train_dataset=ds_tr, eval_dataset=ds_te,
        data_collator=DataCollatorWithPadding(tokenizer=tok), compute_metrics=metricas)
    trainer.train()

    pred = trainer.predict(ds_te)
    yhat = np.argmax(pred.predictions, axis=1)
    ynames = [id2lab[i] for i in pred.label_ids]
    phat = [id2lab[i] for i in yhat]
    print('\n=== BETO — %s — test = 180 gold adjudicado ===' % tarea)
    print(classification_report(ynames, phat, digits=3, zero_division=0))
    print('Macro-F1: %.3f' % f1_score(ynames, phat, average='macro', zero_division=0))
    crit = CRITICO[tarea]
    yt = np.array(ynames); yp = np.array(phat)
    pos = yt == crit
    if pos.sum():
        fn = int(np.sum(pos & (yp != crit)))
        rec = recall_score(yt, yp, labels=[crit], average='macro', zero_division=0)
        print(">>> Clase crítica '%s': recall = %.3f | falsos negativos = %d de %d"
              % (crit, rec, fn, int(pos.sum())))
    trainer.save_model(salida)
    tok.save_pretrained(salida)
    print('Modelo guardado en:', salida)
    return trainer

## 4. Entrenar BETO — Categoría (7 clases)

In [ ]:
entrenar_tarea('categoria', con_sintetico=True, epocas=4)

## 5. Entrenar BETO — Urgencia (3 niveles)

In [ ]:
entrenar_tarea('urgencia', con_sintetico=True, epocas=4)

## 6. Descargar los modelos entrenados

Comprime ambas carpetas y descarga el zip para servir BETO desde el microservicio Python.

In [ ]:
# Re-empaqueta SOLO el modelo final (sin checkpoints ni optimizer): ~880 MB en vez de 9 GB.
# Los checkpoint-*/ traen optimizer.pt (~880 MB c/u), inutiles para servir la inferencia.
from google.colab import drive
import os, shutil

drive.mount('/content/drive')

!rm -f beto_modelos.zip
!zip -qr beto_modelos.zip beto_categoria beto_urgencia -x "*/checkpoint-*/*"

destino = '/content/drive/MyDrive/SIREC'
os.makedirs(destino, exist_ok=True)
shutil.copy('beto_modelos.zip', destino + '/beto_modelos.zip')

print('OK -> guardado en Drive:', destino + '/beto_modelos.zip')
print('Tamaño MB:', round(os.path.getsize('beto_modelos.zip') / 1e6, 1))
!unzip -l beto_modelos.zip | tail -15